<a href="https://colab.research.google.com/github/CodeHunterOfficial/ArabovMKDeep/blob/main/MLF-2026/%D0%97%D0%B0%D0%B4%D0%B0%D0%BD%D0%B8%D0%B5_%D0%B4%D0%BB%D1%8F_%D1%81%D1%82%D1%83%D0%B4%D0%B5%D0%BD%D1%82%D0%BE%D0%B2_%D0%BF%D0%BE_MLF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 📚 Задание для студентов: Разработка интеллектуальной системы прогнозирования временных рядов с мультиагентной архитектурой, Agentic RAG и оценкой качества



## 🎯 Цель проекта

Создать **интеллектуальную систему прогнозирования временных рядов**. Пользователь загружает ряд и указывает горизонт. Система анализирует данные, подбирает оптимальную стратегию (от классических статистических моделей до фундаментальных TS-моделей), обучает/инференсит, диагностирует остатки и предоставляет обоснованный результат.

**Система должна включать:**
- Мультиагентную архитектуру (минимум 3 агента + Supervisor), реализованную на чистом Python (без LangChain/LangGraph) или с использованием легковесных оркестраторов (например, PydanticAI).
- **Agentic RAG** с гибридным поиском и контекстным ретривалом для подбора похожих кейсов.
- Поддержку локальных LLM (Ollama/vLLM) и внешних API с обязательным использованием **Tool Calling (Function Calling)** для агентов-планировщиков, с разумными fallback-механизмами.
- REST API (FastAPI), асинхронность, фоновые задачи.
- Human-in-the-Loop (HITL) для сложных кейсов.
- Строгую систему оценки качества (метрики, сравнение режимов, оценка траекторий агентов).
- Сквозное логирование (Observability) с трассировкой шагов и метриками LLM.

**Дополнительный контекст:** хотя основное внимание уделено временным рядам, архитектура должна быть спроектирована так, чтобы в будущем её можно было расширить на табличные данные (регрессия, классификация) без переписывания агентов и пайплайнов. Для этого разделяйте доменную логику (временные ряды) и общий оркестрационный слой (агенты, RAG, состояние). Ниже приведены конкретные рекомендации, как это реализовать.

**Главное правило:** Студенты не используют «тяжелые» фреймворки для агентов. Вся логика оркестрации и работы с состоянием (State) прозрачна и контролируется разработчиком.

---

## 🏗️ Архитектура проекта (Исправленная)

```text
project/
├── backend/
│   ├── agents/            # Агенты и супервизор
│   │   ├── base.py        # Абстрактный класс Agent (строгая типизация state)
│   │   ├── data_analyst.py
│   │   ├── model_selector.py # Работает ТОЛЬКО через Tool Calling
│   │   ├── validator.py   
│   │   ├── critic.py      # LLM-as-a-Judge
│   │   ├── editor.py      # Опционально: агент-редактор гиперпараметров
│   │   └── supervisor.py  
│   ├── api/               # FastAPI
│   │   ├── routes.py      # Эндпоинты (см. раздел API)
│   │   └── schemas.py     
│   ├── core/
│   │   ├── config.py      
│   │   ├── llm/           # LLMManager (роутинг запросов)
│   │   │   ├── manager.py
│   │   │   ├── providers/ # ollama.py, vllm.py, hf_local.py, api.py
│   │   │   └── tools.py   # Реестр инструментов (Tool Calling)
│   │   ├── metrics.py     
│   │   └── models.py      # Pydantic-модели (включая State)
│   ├── timeseries/        # Доменная логика
│   │   ├── preprocessing.py # Ресемплинг, стационарность
│   │   ├── classical.py   # ARIMA, ETS, Prophet
│   │   └── foundation.py  # 🆕 Интеграция TS Foundation Models (Chronos, TimesFM) — бонус
│   ├── data/              # Только работа с данными и хранилищами
│   │   ├── knowledge_base.py
│   │   ├── cache.py       
│   │   └── embeddings.py  # Синглтон эмбеддеров
│   ├── db/                # SQLAlchemy, миграции (Alembic)
│   ├── evaluation/        # Оценка качества и метрик
│   ├── hitl/              # Human-in-the-Loop
│   ├── retrieval/         # BM25, Qdrant/FAISS, Hybrid Retriever
│   ├── utils/             # Логирование, безопасность, async-утилиты
│   └── worker/            # Celery / ARQ / Taskiq
├── frontend/              # Chainlit или Gradio
├── scripts/               # Генерация данных, run_eval.py
├── tests/                 # pytest, pytest-asyncio
├── infrastructure/        # Docker (только для Prod/CI)
├── pyproject.toml         # Poetry / uv
└── run.py
```

---

## 🤖 Требования к мультиагентной системе

### 1. Управление состоянием (State)
- `state` передается между агентами **только по ссылке** (ID в БД, путь к файлу, ключ в Redis). Передача тяжелых объектов (DataFrame) в payload запрещена.
- State должен быть строго типизирован (Pydantic). Пример модели:

```python
from pydantic import BaseModel, Field
from typing import Optional, Literal, Dict, Any

class ForecastState(BaseModel):
    series_id: str
    horizon: int
    freq: Optional[str] = None
    task_type: Literal["timeseries", "tabular"] = "timeseries"  # тип задачи
    metadata: Dict[str, Any] = Field(default_factory=dict)      # доп. информация (например, кол-во признаков для таблиц)
    target_column: Optional[str] = None  # для табличных задач – имя целевой колонки; для временных рядов обычно None
    model_type: Optional[str] = None          # Например: "arima", "prophet", "xgboost"
    # Формат зависит от модели: {'order': (1,1,1)} для ARIMA, {'seasonality_mode': 'additive'} для Prophet и т.д.
    model_params: Optional[dict] = None
    predictions_path: Optional[str] = None    # путь к файлу/хранилищу
    metrics: Optional[dict] = None
    diagnostics: Optional[dict] = None        # результаты ACF/PACF, тестов
    validation_passed: bool = False
    critic_score: Optional[float] = None
    critic_reasoning: Optional[str] = None
    editor_changed: bool = False
    iteration: int = 0
    max_iterations: int = 3
    log_id: Optional[str] = None              # ссылка на reasoning_log
    hitl_requested: bool = False              # Флаг, что требуется вмешательство пользователя
    hitl_resolution: Optional[dict] = None    # Решение пользователя (например, выбор другой модели)
```

(Допускается расширение полей в зависимости от реализации.)

### 2. Агенты и Tool Calling (Критическое изменение)

- **ModelSelector НЕ генерирует сырой Python-код.** Это антипаттерн, ведущий к галлюцинациям и уязвимостям.
- Агент должен использовать **Tool Calling (Function Calling)**. У студента должен быть реализован *Реестр инструментов* (например, `fit_arima`, `fit_prophet`, `fit_xgboost`, опционально `infer_chronos`). LLM возвращает JSON с названием инструмента и его аргументами (гиперпараметрами).
- **Обязательный минимум:** реестр должен содержать не менее **трёх классических методов** (например, ARIMA, Prophet, XGBoost/LightGBM). Инструменты для **TS Foundation Models (Chronos, TimesFM, Moirai) являются бонусом** и не обязательны для базовой реализации.
- **Расширяемость на табличные данные:** Для демонстрации универсальности архитектуры рекомендуется добавить в реестр хотя бы один инструмент для табличных задач (например, `fit_tabular_xgboost` или `fit_random_forest`), принимающий на вход матрицу признаков X и целевую переменную y. Это покажет, что ModelSelector работает с абстрактными инструментами, а не зашит только во временные ряды.
- **Реалистичность Tool Calling на слабых моделях:** Модели малого размера (SmolLM2, Phi-3.5-mini) не всегда стабильно возвращают корректный JSON по схеме. Поэтому:
  - Для Tool Calling **рекомендуется использовать Qwen2.5 (7B/14B), Llama-3.2/3.3 (8B)** или аналогичные модели с хорошей поддержкой function calling.
  - Для слабых моделей допускается fallback-механизм: сначала попытка строгого function calling; при неудаче – парсинг обычного текстового ответа с последующей валидацией и, при необходимости, повторным запросом с уточнением формата.
  - **Максимальное число повторных попыток при fallback — 3.** Если после трёх попыток LLM не вернула валидный вызов инструмента, используется заранее заданное правило по умолчанию (например, выбирается SARIMA с автоподбором порядка через `auto_arima`).
  - Слабые модели (≤1B) можно использовать для простых генеративных задач (например, Critic или генерация текстовых объяснений), где не требуется строгий JSON.

- **Набор агентов:**
  - **DataAnalyst**: Парсинг, приведение к `DatetimeIndex`, проверка регулярности, ресемплинг, STL-декомпозиция. При `task_type == "tabular"` агент пропускает приведение к `DatetimeIndex`, ресемплинг и STL-декомпозицию, но выполняет проверку типов признаков, импутацию пропусков, one-hot encoding для категориальных переменных и базовую статистику. Если задан `target_column`, отделяет признаки от целевой переменной.
  - **ModelSelector**: Анализирует EDA, использует RAG для поиска похожих кейсов, вызывает LLM с Tool Calling для выбора модели и гиперпараметров из Реестра.
  - **Validator**: Формальная проверка (NaN, диапазоны, длина горизонта, статистические тесты остатков). Для табличных данных – проверка корректности предсказаний (форма, отсутствие NaN, адекватность метрик).
  - **Critic**: LLM-as-a-Judge. Оценивает адекватность выбора модели и интерпретируемость.
  - *(Опционально)* **Editor**: Если Validator/Critic недовольны, предлагает изменить гиперпараметры или добавить лаги/признаки.

### 3. Супервизор (Supervisor)
- Оркеструет цикл. Ограничивает итерации (`max_iterations`).
- Ведет **Reasoning Log** (трассировку): кто вызвал, какой Tool был использован, вход/выход, время, токены.
- **Reasoning Log должен поддерживать иерархическую структуру (parent-child relationship между шагами)**, чтобы можно было восстановить дерево рассуждений, а не просто плоский список событий. Для этого в каждой записи лога должно быть поле `parent_id` (или `span_id` / `trace_id`).
- Логи сохраняются в БД и доступны через API.
- **Human-in-the-Loop (HITL):** Если после `max_iterations` цикл не сошёлся, или Validator/Critic выставили критические замечания, Supervisor приостанавливает задачу и запрашивает решение пользователя через API/UI. Пользователь может:
  - выбрать другую модель (из списка доступных),
  - изменить горизонт,
  - принять текущий результат.
  Решение пользователя фиксируется в `hitl_resolution` и передаётся агенту Editor (если есть) для корректировки, либо напрямую изменяет State.

---

## 🔍 Требования к Agentic RAG и Эмбеддингам

1. **Гибридный поиск и Контекстный ретривал**
   - BM25 (лексический) + Векторный поиск.
   - Использование **Reciprocal Rank Fusion (RRF)** или learning-to-rank.
   - *2026 Trend:* Внедрение **Contextual Retrieval** (добавление краткого контекстного описания к чанкам перед эмбеддингом для улучшения семантического поиска).

2. **Эмбеддинги временных рядов**
   - **Базовое требование:** Извлечение статистических признаков (`tsfresh` или hand-crafted: тренд, сезонность, энтропия, автокорреляции) + **снижение размерности** (PCA / UMAP) до 128-256 dims.
   - **Бонус (2026 Trend):** Использование предобученных представлений рядов (например, через projection head от TimesFM или Chronos) для получения dense-векторов.
   - **Замечание по расширяемости:** При расширении на табличные данные мета-фильтрация должна поддерживать атрибуты: тип задачи (регрессия/классификация), количество признаков, доля пропусков. Для временных рядов поиск похожих кейсов опирается на статистические признаки (тренд, сезонность, энтропия). Убедитесь, что ваша схема метаданных в векторной БД учитывает оба типа.

   **Адаптивность извлечения признаков:** Реализуйте `FeatureExtractor` как интерфейс с разными реализациями для `timeseries` и `tabular` (например, через паттерн «стратегия» и конфигурацию). Для табличных данных признаками могут быть: корреляции признаков с целевой переменной, базовые статистики распределений (среднее, дисперсия, асимметрия), количество категорий, доля пропусков. Для временных рядов — тренд, сезонность, автокорреляции, энтропия. Выбор конкретного экстрактора определяется полем `task_type`.

3. **Векторная БД**
   - Qdrant (для продакшена) или FAISS/Chroma (для локальной разработки). Поддержка фильтров по метаданным (частота, тип ряда, тип задачи, target_column и т.д.).
   - **Пример метаданных для табличного кейса** (хранится вместе с вектором):
     ```json
     {
       "task_type": "tabular",
       "num_features": 15,
       "num_rows": 10000,
       "target_column": "price",
       "problem_type": "regression",
       "missing_rate": 0.02,
       "has_categorical": true
     }
     ```
     Это позволяет использовать фильтры Qdrant/FAISS для ограничения поиска только релевантными типами задач.

---

## 🧠 Поддержка LLM и Инференс (Стандарты 2026)

1. **Локальные LLM (Основной вариант)**
   - Использование **Ollama** или **vLLM** (для продвинутого инференса).
   - Модели: `Qwen2.5` (7B/14B), `Llama-3.2/3.3` (8B), `SmolLM2` (для слабых машин).
   - **Обязательное требование:** Поддержка **Tool Calling / Structured Output** (модель должна уметь возвращать строгий JSON по схеме Pydantic). Для слабых моделей допускается fallback (см. выше).

2. **HuggingFace Transformers (Запасной/Локальный)**
   - Только **Causal Language Models** (генеративные).
   - Примеры: `Qwen/Qwen2.5-0.5B-Instruct`, `HuggingFaceTB/SmolLM2-360M-Instruct`, `microsoft/Phi-3.5-mini-instruct`.
   - *Запрещено:* Использование Encoder-only (BERT) или Seq2Seq (T5) без явного обоснования и адаптации.

3. **Внешние API (Бонус)**
   - OpenAI, Anthropic, Groq, OpenRouter. Унифицированный интерфейс с retry-логикой и fallback.

4. **Подсчёт токенов для локальных LLM:** Если API локальной LLM (Ollama, vLLM) не возвращает точное количество токенов, студент должен использовать локальный токенизатор (например, `tiktoken` или `transformers.AutoTokenizer`) для приблизительного подсчёта входных и выходных токенов. Эти данные нужны для оценки стоимости агентного подхода.

---

## 📊 Датасеты и источники данных

Для обучения, создания базы знаний и оценки системы необходимо использовать **открытые наборы данных**. Ниже перечислены рекомендуемые источники.

### Основные наборы для временных рядов
| Название | Описание | Источник |
|----------|----------|----------|
| **M4** | 100 000 рядов различной частоты (ежечасные, дневные, недельные, месячные, квартальные, годовые) из разных доменов (финансы, демография, экономика). | GitHub: `M4-methods`, пакет `M4comp2018` |
| **M5** | Иерархические ряды продаж Walmart (daily, с календарными признаками). | Kaggle: `m5-forecasting-accuracy`, GitHub: `M5-methods` |
| **Monash Time Series Forecasting Repository** | Большой репозиторий наборов из разных областей: энергетика, транспорт, здравоохранение, погода и др. | Сайт: `monash.edu/business/forecasting` |
| **Electricity Load Diagrams** | Потребление электроэнергии (15-минутные интервалы, 370 клиентов). | UCI ML Repository |
| **Jena Climate** | Метеорологические данные (температура, давление, влажность и др.) с 10-минутным интервалом. | Kaggle: `jena-climate`, UCI |
| **Air Quality** | Данные о качестве воздуха (концентрации газов, метеопоказатели) с почасовой частотой. | UCI ML Repository |
| **Traffic** | Загруженность дорог (почасовая, 862 точки). | UCI ML Repository |

### Для табличных задач (расширение)
| Название | Описание | Источник |
|----------|----------|----------|
| **California Housing** | Регрессия: предсказание цен на жильё. | `sklearn.datasets.fetch_california_housing` |
| **Adult (Census Income)** | Классификация: предсказание дохода >50K. | UCI ML Repository |
| **Titanic** | Классификация: выживаемость пассажиров. | Kaggle: `titanic` |
| **Bank Marketing** | Классификация: подписка на депозит. | UCI ML Repository |
| **Hugging Face Datasets** | Множество готовых датасетов для регрессии и классификации. | `datasets.load_dataset()` |

### Использование Kaggle
- Для доступа к данным Kaggle можно использовать API: `kaggle datasets download -d <dataset-name>`.
- В проекте рекомендуется написать скрипт `scripts/download_data.py`, который автоматизирует загрузку нужных датасетов.

### Использование Hugging Face Datasets
- Для временных рядов можно использовать `datasets` с готовыми наборами, например `monash_tsf`, `m4`, `m5`.
- Для табличных данных – `datasets` имеет множество вариантов.
- Пример загрузки:
  ```python
  from datasets import load_dataset
  ds = load_dataset("monash_tsf", "electricity")
  ```

### Требование к разнообразию признаков
> **Важно:** при выборе датасетов (особенно табличных) старайтесь, чтобы **количество признаков было не менее 5** (в идеале 10 и более). Это гарантирует, что система будет обрабатывать многомерные зависимости, а не тривиальные случаи с 2–3 переменными.  
> Для временных рядов допускаются как одномерные, так и многомерные наборы (с несколькими временными рядами или экзогенными переменными). Если ряд одномерный, можно добавить внешние признаки (например, календарные, погодные), чтобы увеличить размерность.  
> В отчёте обязательно укажите число признаков для каждого использованного датасета.

### Примеры датасетов, загружаемых через `kagglehub`

```python
import kagglehub

# Климатический временной ряд
path_climate = kagglehub.dataset_download("sumanthvrao/daily-climate-time-series-data")

# Энергопотребление (временной ряд)
path_energy = kagglehub.dataset_download("vitthalmadane/energy-consumption-time-series-dataset")

# Табличные данные: образование (регрессия/классификация)
path_education = kagglehub.dataset_download("saadaziz1985/bachelors-or-higher-degree-data-of-usa")

# Табличные данные: демография (регрессия)
path_population = kagglehub.dataset_download("saadaziz1985/population-collapse")
```

| Датасет (kagglehub path) | Тип | Ориентировочное число признаков |
|--------------------------|-----|----------------------------------|
| `sumanthvrao/daily-climate-time-series-data` | Временной ряд (погода) | 4–6 (можно добавить лаги и скользящие статистики) |
| `vitthalmadane/energy-consumption-time-series-dataset` | Временной ряд (энергопотребление) | 2–5 (добавить календарные признаки) |
| `saadaziz1985/bachelors-or-higher-degree-data-of-usa` | Табличный (регрессия/классификация) | 5–10+ (зависит от версии) |
| `saadaziz1985/population-collapse` | Табличный (регрессия) | 3–7 (можно дополнить синтетическими признаками) |

**Совет:** если у выбранного датасета мало признаков, создайте дополнительные (лаги, взаимодействия, агрегаты по времени и т.п.), чтобы достичь минимального порога в 5 признаков. Это также улучшит качество RAG-поиска, так как метаданные будут богаче.

---

## 📊 Оценка качества и Сравнение (run_eval.py)

### 1. Метрики
- **Точность (для временных рядов и регрессии):** MAE, RMSE, MAPE, MASE, WAPE.
- **Для табличных задач:**
  - Если задача регрессии (`task_type == "tabular"` и целевая переменная непрерывна) — MAE/RMSE.
  - Если задача классификации (целевая переменная категориальна) — Accuracy, F1, ROC-AUC.
- **Диагностика остатков (временные ряды):** Тест Льюнга-Бокса (Ljung-Box) на автокорреляцию остатков, тест Шапиро-Уилка на нормальность, анализ ACF/PACF.
- **Статистические тесты для сравнения моделей:**
  - **Тест Диеболда-Мариано (Diebold-Mariano)** для сравнения точности прогнозов двух моделей на одном ряде.
  - **Тест знаковых рангов Уилкоксона (Wilcoxon signed-rank)** для сравнения распределений ошибок по нескольким рядам.
  - **Bootstrap-доверительные интервалы** для метрик.
  - Применение этих тестов обязательно при сравнении режимов (baseline vs +rag vs +agents) в отчёте.

- **Агентные метрики (2026 Trend):** Оценка траектории — **среднее количество итераций Supervisor до достижения критерия успеха**, **доля случаев, потребовавших HITL-подтверждение**, а также **стоимость** (время выполнения, количество LLM-токенов). Эти метрики вычисляются на основе данных из Reasoning Log (иерархического).

### 2. Сбор метрик на подвыборках разного размера
Для оценки влияния объёма данных на качество прогнозов необходимо для каждого ряда (или набора рядов) строить модели на **подвыборках исходного ряда**, например:
- 10% последних наблюдений (или 10% временного интервала),
- 20%,
- 50%,
- 100% (весь ряд).

Это позволит построить кривую «точность vs размер обучающей выборки» и проанализировать, сколько данных достаточно для разных моделей. Для табличных данных аналогично используются стратифицированные выборки (с сохранением пропорций классов для классификации). Результаты должны быть представлены в отчёте.

### 3. Скрипт `scripts/run_eval.py`
- Параметры:
  - `--num-series [20, 50, 100]` – количество рядов для оценки.
  - `--horizons [10, 20, 50]` – горизонты прогнозирования.
  - `--data-fractions [0.1, 0.2, 0.5, 1.0]` – доли данных для обучения (по умолчанию [1.0]).
- **Режимы:** `baseline` (классика без LLM), `+rag`, `+agents` (полный цикл), `+foundation` (использование TS Foundation Models, если доступно).
- **Проверка данных:** Автоматическая проверка достаточности данных для MASE и сезонных моделей.
- Логи работы скрипта пишутся в `eval.log` для пост-мортем анализа.
- **Подсчёт агентных метрик:** для каждого запуска в режиме `+agents` фиксируются: количество итераций, факт HITL-вмешательства, время выполнения, количество токенов (если доступно). Итоговые средние значения сохраняются в результатах.

---

## 📥 Сбор эталонных данных (База знаний)

- **База знаний (1000+ записей):** Разрешается и **поощряется синтетическая генерация**. Студент пишет скрипт, который берет открытые ряды (M4/M5, Monash, Jena Climate и др.), прогоняет их через сетку классических моделей, находит лучшую для каждого ряда и сохраняет это как «успешный кейс» в БД.
- **Формат записи успешного кейса (пример):**
  ```json
  {
    "series_id": "M4_123",
    "task_type": "timeseries",
    "features": { ... },              // статистические признаки
    "embedding": [ ... ],             // вектор 128-256 dims
    "best_model": "arima",
    "hyperparams": {"order": [1,1,1]},
    "metrics": {"smape": 0.12, "mase": 0.85},
    "description": "Ряд с ярко выраженной недельной сезонностью и слабым трендом. ARIMA(1,1,1) сработала лучше Prophet за счёт отсутствия выбросов."
  }
  ```
  Для табличных кейсов добавить поле `target_column` и метрики классификации/регрессии.
  В RAG используются как текстовые описания (description), так и структурированные признаки с эмбеддингами.
- **Справочник (500+ терминов):** Определения, формулы, советы по тюнингу.

---

## 🛠️ Backend и Инфраструктура (DevEx & Prod)

1. **Разделение окружений (Критически важно для студентов)**
   - **Local Dev:** Разрешается использовать SQLite, in-memory кэш, FAISS вместо Qdrant, моки вместо Ollama. Это спасет слабые ноутбуки.
   - **Production/CI:** Полный `docker-compose` (PostgreSQL, Redis, Qdrant, Ollama/vLLM, FastAPI, Worker).

2. **Безопасность выполнения (Sandbox)**
   - Если в проекте все же есть выполнение динамического кода (не рекомендуется, но возможно для бонуса): использовать **пул прогретых Docker-контейнеров** (например, E2B или кастомный пул) или **WASM / RestrictedPython**. Запуск нового контейнера на каждый запрос недопустим.

3. **Фоновые задачи**
   - Celery или современные альтернативы (Taskiq / ARQ). Задачи: переиндексация RAG, пакетный инференс, очистка кэша.

4. **Observability (усилено)**
   - В обязательном порядке реализуется логирование всех вызовов LLM: модель, параметры, количество токенов (если доступно), latency, статус успеха.
   - Эти данные сохраняются в structured logs (или в БД) и доступны для анализа. Использование внешних систем (Langfuse, Phoenix) — бонус.

5. **API (минимальный набор эндпоинтов):**
   - `POST /api/v1/forecast/` — загрузка ряда (или ссылка на файл) и параметров (горизонт, частота, task_type, target_column при необходимости). **Возвращает HTTP 202 Accepted** и `task_id` для последующего опроса статуса.
   - `GET /api/v1/forecast/{task_id}/status` — статус задачи (pending, running, waiting_for_human, completed, failed).
   - `GET /api/v1/forecast/{task_id}/result` — результат прогноза, метрики, выбранная модель, путь к файлу с прогнозом.
   - `GET /api/v1/forecast/{task_id}/reasoning` — дерево рассуждений (Reasoning Log).
   - `POST /api/v1/forecast/{task_id}/hitl` — решение пользователя для HITL (например, выбранная модель или подтверждение).

---

## 🖥️ Frontend (UI для агентов)

**Streamlit больше не является рекомендуемым выбором** для сложных агентных систем из-за проблем с асинхронностью и состоянием.
- **Рекомендация 2026:** Использовать **Chainlit** или **Gradio**. Они «из коробки» поддерживают стриминг ответов агентов, отображение промежуточных шагов (Reasoning logs), кнопки HITL и работу с асинхронным бэкендом.
- Если используется кастомный фронтенд (React/Vue), он должен уметь отображать дерево рассуждений агентов в реальном времени.

---

## 🧪 Тестирование

- **Покрытие:** ≥ 70% (`pytest-cov`).
- **Обязательные тесты:**
  - Юнит-тесты для каждого агента (с моками LLM).
  - Тесты на корректность работы Tool Calling (парсинг JSON от LLM, включая fallback).
  - Тесты для `timeseries/preprocessing.py` (проверка ресемплинга и стационарности).
  - Интеграционные тесты API.
- Использование `pytest-asyncio` и `respx` / `httpx` для мока внешних вызовов.

---

## 📋 Критерии оценки

| Критерий | Вес | Описание |
|----------|-----|----------|
| **Архитектура и Tool Calling** | 25% | Чистота кода, строгая типизация State, отсутствие сырой генерации кода, грамотный Реестр инструментов. |
| **Функциональность и RAG** | 25% | Работа агентов, качество гибридного поиска, интеграция TS-моделей (если есть). |
| **Оценка качества (Eval)** | 15% | Корректность `run_eval.py`, разделение горизонтов, агентные метрики, статистические тесты, анализ подвыборок. |
| **Тестирование и CI/CD** | 15% | Покрытие, моки, наличие GitHub Actions (опционально). |
| **Документация и Отчет** | 10% | README, описание архитектуры, обоснование выбора моделей. |
| **Инфраструктура и DevEx** | 10% | Наличие легкого локального режима и полноценного Docker-compose. |

**Бонусы (до +20%):**
- Интеграция **TS Foundation Models** (Chronos, TimesFM, Moirai) в качестве инструментов или бейзлайнов (+5%).
- Использование **Contextual Retrieval** или GraphRAG для базы знаний (+4%).
- Продвинутый UI с визуализацией дерева рассуждений агентов в реальном времени (+3%).
- Статистическая значимость в `run_eval.py` (bootstrap, paired t-test) (+3%).
- Интеграция с системами Observability для LLM (Langfuse, Phoenix) (+3%).
- Использование пула прогретых sandbox-контейнеров или WASM (+2%).
- **Реализация автоматического feature engineering (генерация лагов, скользящих статистик, полиномиальных признаков) как отдельного агента или инструмента в реестре (+2%).**

---

## 📦 Итоговый результат (Что сдает студент)

1. Репозиторий с кодом (разделенный на `local` и `prod` конфигурации).
2. README с инструкциями по быстрому старту (локально) и запуску в Docker.
3. Скрипты генерации и импорта синтетической базы знаний.
4. Скрипт `run_eval.py` с раздельными параметрами, агентными метриками, статистическими тестами и анализом подвыборок.
5. Отчет (PDF/MD) с анализом результатов оценки: какие режимы (baseline/rag/agents) и на каких горизонтах работают лучше, анализ стоимости (время/токены) агентного подхода, зависимость точности от объёма данных, результаты статистических тестов.

---
**Удачи!** Помните: в 2026 году ценится не «магия» промптов, а надежная инженерия, строгая типизация, контролируемые агенты и глубокая работа с данными.


## Поэтапный план разработки (15 недель)

### Этап 1: Выбор темы и настройка окружения

**Цель:** Определиться с предметной областью (временные ряды, табличные данные) и подготовить рабочее место.

**Задачи:**
- Выбрать конкретную задачу прогнозирования (например, прогноз энергопотребления, погоды, продаж) и обосновать её.
- Сформировать команду (2–3 человека), распределить роли.
- Создать удалённый репозиторий (GitHub/GitLab).
- Настроить Poetry: создать `pyproject.toml` со всеми зависимостями (перечислены в общем задании: fastapi, sqlalchemy, pydantic, pandas, numpy, scikit-learn, statsmodels, prophet, xgboost, transformers, sentence-transformers, qdrant-client, redis, celery, pytest и др.).
- Добавить `.gitignore`, черновик README (цель, стек, краткое описание).
- Определить предварительный список датасетов (M4, M5, Monash, Jena Climate и т.д.) и метрик (MAE, RMSE, MAPE, MASE, WAPE, а также Accuracy/F1 для классификации).

**Результат:**  
Репозиторий с Poetry, выбранной темой и первичным README.

**Критерии готовности:**
- [ ] Тема выбрана и обоснована.
- [ ] Poetry устанавливает зависимости без ошибок.
- [ ] README содержит краткое описание проекта и список метрик.

---

### Этап 2: Проектирование архитектуры и каркас приложения

**Цель:** Создать скелет проекта и базовые модули.

**Задачи:**
- Спроектировать структуру папок в соответствии с заданием (backend/agents, backend/api, backend/core, backend/timeseries, backend/data, backend/retrieval, backend/evaluation, backend/hitl и т.д.).
- Реализовать `backend/core/config.py` (pydantic-settings): БД, LLM (Ollama/vLLM/API), Qdrant, Redis, пути к данным.
- Создать `backend/core/models.py` (Pydantic-модели, включая `ForecastState` с полями `task_type`, `target_column`, `metadata` и др.).
- Создать `backend/api/main.py` (FastAPI, CORS, `/healthz`).
- Написать `run.py` для запуска uvicorn.
- Настроить логирование (`backend/utils/logging.py`).

**Результат:**  
Приложение запускается, отвечает на `/healthz`, структура папок готова.

**Критерии готовности:**
- [ ] Структура папок соответствует архитектуре.
- [ ] Конфигурация читается из переменных окружения.
- [ ] Логирование работает (консоль + файл).

---

### Этап 3: База данных

**Цель:** Настроить подключение к БД и создать модели данных.

**Задачи:**
- Реализовать `backend/db/session.py` (async SQLAlchemy).
- Создать `backend/db/models.py`:
  - `users` (id, email, username, hashed_password, role, is_active, created_at)
  - `knowledge_base` (id, series_features, embedding, best_model, hyperparams, metrics, description, task_type, target_column, created_at)
  - `glossary` (id, term, definition, category, created_at) — справочник терминов
  - `cache` (id, input_key, output, hits, created_at)
  - `evaluation_results` (id, model_name, mode, sample_size, metrics_json, evaluation_time, created_at)
  - `model_comparisons` (id, models_compared, sample_size, results_json, winner, created_at)
  - **`reasoning_logs`** (id, request_id, agent_name, input_summary, output_summary, model_name, parameters, started_at, finished_at, status, parent_id) — для иерархического лога рассуждений
  - **`llm_calls`** (id, request_id, agent_name, model_name, provider, prompt_hash, response_hash, tokens_used, latency_ms, status, created_at) — для трассировки вызовов LLM
- Настроить автоматическое создание таблиц при старте.
- Подключить PostgreSQL в docker-compose (локально можно SQLite).
- Протестировать подключение.

**Результат:**  
БД инициализируется, все таблицы создаются.

**Критерии готовности:**
- [ ] Приложение подключается к БД.
- [ ] Все таблицы, включая `reasoning_logs` и `llm_calls`, создаются автоматически.
- [ ] Модели описаны корректно.

---

### Этап 4: Аутентификация и авторизация

**Цель:** Реализовать регистрацию, вход и систему ролей.

**Задачи:**
- Создать `backend/utils/security.py` (bcrypt, JWT).
- Реализовать `backend/api/deps.py` (зависимости `get_current_user`, `require_admin`, `require_editor`).
- Создать роуты `backend/api/routes/auth.py`:
  - `POST /auth/register`
  - `POST /auth/login`
  - `GET /auth/me`
- Добавить Pydantic-модели для запросов/ответов.
- Написать юнит-тесты (мок-сессия).

**Результат:**  
Работающая аутентификация с ролями.

**Критерии готовности:**
- [ ] Пароли хранятся в виде bcrypt-хеша.
- [ ] JWT выпускается и проверяется.
- [ ] Тесты проходят.

---

### Этап 5: Базовый класс агента и Supervisor

**Цель:** Заложить основу мультиагентной системы.

**Задачи:**
- Создать `backend/agents/base.py` (абстрактный `Agent` с `async def process(state)` и строгой типизацией).
- Создать `backend/agents/supervisor.py`:
  - хранит ссылки на агентов,
  - лениво инициализирует,
  - организует цикл до выполнения условий или достижения `max_iterations`,
  - ведёт иерархический `reasoning_log` (сохраняет записи в таблицу `reasoning_logs` с полем `parent_id`).
- Написать юнит-тесты (mock-агенты).

**Результат:**  
Рабочий Supervisor, управляющий агентами и сохраняющий лог рассуждений.

**Критерии готовности:**
- [ ] Базовый класс реализован.
- [ ] Supervisor корректно выполняет цикл.
- [ ] `reasoning_log` заполняется и сохраняется в БД (с иерархией).
- [ ] Тесты проходят.

---

### Этап 6: Агент DataAnalyst и предобработка временных рядов

**Цель:** Реализовать агента для анализа данных и подготовки признаков.

**Задачи:**
- Создать `backend/agents/data_analyst.py`.
- Реализовать логику для `task_type == "timeseries"`: парсинг дат, проверка регулярности, ресемплинг, STL-декомпозиция, проверка стационарности (ADF тест).
- Реализовать логику для `task_type == "tabular"`: проверка типов, импутация пропусков, one-hot encoding, базовая статистика.
- Записывать в `state`: `diagnostics`, `metadata` (число признаков и т.д.).
- Написать юнит-тесты для обоих типов.

**Результат:**  
DataAnalyst корректно подготавливает данные.

**Критерии готовности:**
- [ ] Для временных рядов выполняется ресемплинг и STL.
- [ ] Для табличных данных выполняется кодирование и импутация.
- [ ] Тесты проходят.

---

### Этап 7: Реестр инструментов (Tool Calling) и агент ModelSelector

**Цель:** Реализовать механизм Tool Calling и агента выбора модели.

**Задачи:**
- Создать `backend/core/llm/tools.py` — реестр инструментов:
  - `fit_arima`, `fit_prophet`, `fit_xgboost` (минимум 3 классических),
  - опционально `fit_tabular_xgboost` или `fit_random_forest` для табличных задач,
  - бонус: `infer_chronos` (TS Foundation Model).
- Реализовать `backend/core/llm/manager.py` с поддержкой Tool Calling (JSON-схемы инструментов).
- Создать `backend/agents/model_selector.py`:
  - использует RAG (пока заглушка) для поиска похожих кейсов,
  - формирует промпт для LLM с описанием инструментов,
  - вызывает LLM (пока заглушка, но с логированием вызова),
  - парсит ответ (JSON) и валидирует выбранный инструмент и параметры,
  - реализует fallback: 3 попытки, затем выбор SARIMA по умолчанию.
- Записывать в `state`: `model_type`, `model_params`.
- Написать юнит-тесты (промпт, парсинг JSON, fallback).

**Результат:**  
ModelSelector выбирает модель через Tool Calling, с fallback при сбое.

**Критерии готовности:**
- [ ] Реестр инструментов содержит минимум 3 метода.
- [ ] Tool Calling возвращает корректный JSON.
- [ ] Fallback срабатывает после 3 неудач.
- [ ] Тесты проходят.

---

### Этап 8: Агент Validator и агент Critic

**Цель:** Реализовать агентов формальной проверки и оценки качества.

**Задачи:**
- Создать `backend/agents/validator.py`:
  - проверка прогнозов на NaN, длину горизонта,
  - статистические тесты остатков (Ljung-Box, Shapiro-Wilk),
  - для табличных задач проверка формы и метрик.
  - Записывать в `state`: `validation_passed`, `diagnostics`.
- Создать `backend/agents/critic.py`:
  - формировать промпт с few-shot примерами для оценки выбора модели,
  - вызывать LLM (пока заглушка, но с логированием),
  - парсить JSON с оценками (например, `critic_score`, `critic_reasoning`),
  - fallback: средние оценки.
  - Записывать в `state`: `critic_score`, `critic_reasoning`.
- Написать юнит-тесты.

**Результат:**  
Validator находит формальные недостатки, Critic оценивает качество.

**Критерии готовности:**
- [ ] Validator выполняет статистические проверки.
- [ ] Critic возвращает оценки и объяснения.
- [ ] Fallback для Critic работает.
- [ ] Тесты проходят.

---

### Этап 9: Сбор и импорт эталонных данных

**Цель:** Подготовить датасеты и создать базу знаний.

**Задачи:**
- Найти/скачать открытые датасеты (M4, M5, Monash, Jena Climate и др.) через `kagglehub` или `datasets`.
- Написать `scripts/generate_knowledge_base.py`:
  - для каждого ряда/набора данных прогнать сетку классических моделей (ARIMA, Prophet, XGBoost),
  - выбрать лучшую модель по метрике (например, MASE),
  - сохранить запись с признаками (статистики, tsfresh), эмбеддингами (PCA/UMAP), описанием, метриками.
- Написать `scripts/import_data.py` (SQLAlchemy, защита от дубликатов).
- Загрузить данные и проверить объёмы (база знаний ≥1000 записей, справочник ≥500 терминов).
- Подготовить отдельный тестовый набор (например, 200 рядов), не пересекающийся с обучающими.

**Результат:**  
В БД ≥1500 записей. Скрипты работают, есть отдельный тестовый набор.

**Критерии готовности:**
- [ ] Датасеты загружены.
- [ ] База знаний сгенерирована и импортирована.
- [ ] Объёмы соответствуют требованиям.
- [ ] Тестовый набор создан.

---

### Этап 10: RAG – индексы и гибридный поиск

**Цель:** Реализовать поиск по базе знаний.

**Задачи:**
- Создать `backend/data/embeddings.py` (синглтон для вычисления эмбеддингов рядов: статистические признаки + PCA/UMAP, бонус — предобученные представления).
- Создать `backend/retrieval/bm25_index.py` (лексический поиск по описаниям).
- Создать `backend/retrieval/vector_index.py` (Qdrant: коллекция, добавление, поиск с фильтрами по метаданным, например `task_type`, `num_features`).
- Создать `backend/retrieval/hybrid_retriever.py` (Reciprocal Rank Fusion).
- Обновить `ModelSelector` для использования гибридного поиска.
- Написать юнит-тесты (BM25, RRF, мок-клиент Qdrant).

**Результат:**  
Гибридный поиск комбинирует лексическое и семантическое сходство.

**Критерии готовности:**
- [ ] Эмбеддинги вычисляются.
- [ ] BM25 индекс работает.
- [ ] Векторный поиск работает с Qdrant.
- [ ] RRF возвращает объединённый список.
- [ ] ModelSelector использует RAG.

---

### Этап 11: Интеграция LLM (Ollama, HuggingFace, внешние API) и логирование вызовов

**Цель:** Подключить реальные LLM, настроить выбор бэкенда и обеспечить полное логирование вызовов.

**Задачи:**
- Реализовать `backend/core/llm/providers/ollama.py` (асинхронный клиент Ollama: HTTP API, таймауты, повторы).
- Реализовать `backend/core/llm/providers/hf_local.py` (обёртка `transformers.pipeline`, асинхронный вызов).
- Реализовать `backend/core/llm/providers/api.py` (клиенты для OpenAI, Anthropic, Groq, унифицированный интерфейс).
- Создать `backend/core/llm/manager.py` (выбор между провайдерами, retry-логика, поддержка Tool Calling).
- **Реализовать модуль логирования вызовов LLM** (например, `backend/utils/llm_logger.py`), который:
  - сохраняет в таблицу `llm_calls` информацию о каждом вызове: агент, модель, провайдер, хэш промпта, хэш ответа, токены (если доступны, иначе использовать `tiktoken` или `AutoTokenizer`), время выполнения, статус;
  - вызывается автоматически из обёрток LLM.
- Обновить агентов (ModelSelector, Critic, Editor) для использования реального LLM (с логированием).
- Написать тесты с моками (httpx, pipeline), включая проверку записи в `llm_calls`.

**Результат:**  
Система обращается к LLM через все бэкенды, автоматически выбирает подходящий, и каждый вызов логируется.

**Критерии готовности:**
- [ ] Клиент Ollama работает.
- [ ] HF pipeline вызывается асинхронно.
- [ ] Внешние API вызываются через унифицированный интерфейс.
- [ ] Агенты переведены на реальные LLM.
- [ ] Все вызовы LLM записываются в таблицу `llm_calls`.
- [ ] Тесты проходят.

---

### Этап 12: Основные API эндпоинты и начало метрик

**Цель:** Связать агентов, RAG и LLM через REST API, начать реализацию метрик.

**Задачи:**
- Создать `backend/core/metrics.py` (MAE, RMSE, MAPE, MASE, WAPE для регрессии; Accuracy, F1, ROC-AUC для классификации; Ljung-Box, Shapiro-Wilk для диагностики).
- Реализовать основной роут `POST /api/v1/forecast/`:
  - принимает файл или ссылку на данные, параметры (горизонт, task_type, target_column),
  - создаёт задачу (возвращает `task_id`, HTTP 202),
  - запускает фоновую задачу (Celery) с полным циклом агентов.
- Добавить роуты:
  - `GET /api/v1/forecast/{task_id}/status`,
  - `GET /api/v1/forecast/{task_id}/result`,
  - `GET /api/v1/forecast/{task_id}/reasoning` (иерархический лог),
  - `POST /api/v1/forecast/{task_id}/hitl` (для HITL).
- Настроить Celery задачу `forecast_task`.
- Написать интеграционные тесты API с `TestClient` и моками.

**Результат:**  
Полноценное API выполняет основную задачу, используя мультиагентную систему и RAG; метрики доступны; логи рассуждений можно получить.

**Критерии готовности:**
- [ ] Основной эндпоинт создаёт задачу.
- [ ] Фоновая задача выполняется.
- [ ] Эндпоинты статуса/результата/логов работают.
- [ ] Метрики реализованы и вызываются.
- [ ] Интеграционные тесты проходят.

---

### Этап 13: Оценка качества – скрипт `run_eval.py`, API сравнения, статистическая значимость (бонус)

**Цель:** Реализовать полноценную систему оценки и сравнения моделей.

**Задачи:**
- Реализовать `scripts/run_eval.py`:
  - параметры: `--num-series`, `--horizons`, `--data-fractions`, `--mode` (baseline, +rag, +agents, +foundation),
  - загрузка данных, выборки 20/50/100,
  - вычисление метрик точности и агентных метрик (итерации, HITL, токены, время),
  - сохранение результатов в JSON/CSV, сводная таблица,
  - **опционально: расчёт статистической значимости (bootstrap, paired t-test, Diebold-Mariano, Wilcoxon) и доверительных интервалов (бонус)**.
- Доработать API:
  - `GET /api/v1/evaluate/compare?models=...&sample_size=...`
  - `POST /api/v1/evaluate/batch`
- Создать Celery задачи:
  - `evaluate_model_task`
  - `compare_models_task`
- Сохранять результаты в `evaluation_results` и `model_comparisons`.
- Написать тесты для скрипта и API (с моками LLM).

**Результат:**  
Работающий скрипт оценки, API сравнения, фоновые задачи оценки.

**Критерии готовности:**
- [ ] Скрипт запускается и выдаёт результаты для разных режимов и размеров.
- [ ] API сравнения возвращает корректные данные.
- [ ] Celery задачи выполняются и сохраняют результаты.
- [ ] (Бонус) Статистическая значимость считается, если реализовано.

---

### Этап 14: Human-in-the-Loop, Active Learning, кэширование и финальное покрытие тестами

**Цель:** Реализовать обратную связь, обработку неоднозначных результатов, оптимизацию и гарантировать качество кода.

**Задачи:**
- Реализовать HITL:
  - `backend/hitl/active_learning.py` (добавление в очередь при низком качестве, получение списка, подтверждение/отклонение).
  - `backend/hitl/feedback.py` (сохранение исправлений).
  - Роуты `/hitl/*` (защищены ролями editor/admin).
  - При подтверждении — обновление базы знаний.
- Настроить кэш (таблица в БД или Redis) для результатов LLM и прогнозов, счётчик попаданий.
- Добавить Celery задачи: переиндексация RAG, очистка кэша, пакетная обработка (если не сделано ранее).
- **Провести финальное тестирование покрытия кода: запустить `pytest --cov`, убедиться, что покрытие ≥70% (или требуемый порог), при необходимости добавить недостающие тесты.**
- Написать юнит-тесты для HITL и кэша.

**Результат:**  
Редакторы могут обрабатывать неоднозначные примеры; кэш ускоряет работу; покрытие тестами соответствует требованиям.

**Критерии готовности:**
- [ ] Очередь неоднозначных примеров работает.
- [ ] Подтверждение обновляет базу знаний.
- [ ] Кэш используется, счётчик попаданий увеличивается.
- [ ] Celery задачи выполняются.
- [ ] Покрытие тестами ≥70% (проверено `pytest-cov`).

---

### Этап 15: Frontend (Chainlit или Gradio), финальная сборка, CI/CD и отчёт

**Цель:** Создать UI, подготовить проект к сдаче, автоматизировать проверки и написать итоговую документацию.

**Задачи:**
- Настроить frontend с использованием **Chainlit** или **Gradio** (рекомендованы вместо Streamlit):
  - страница входа/регистрации,
  - основная страница для загрузки ряда/таблицы и запуска прогноза,
  - отображение статуса задачи и результатов,
  - страница HITL (очередь, подтверждение/отклонение),
  - **страница оценки качества**: форма запуска (выбор моделей, размеров, режимов), таблицы сравнения, графики (Plotly),
  - **страница просмотра логов рассуждений и LLM-вызовов**: выбор задачи, пошаговое отображение действий агентов и трассировки вызовов.
- Написать `api_client.py` для взаимодействия с FastAPI.
- Создать Dockerfile и docker-compose.yml со всеми сервисами (app, worker, beat, db, redis, qdrant, ollama, nginx).
- Настроить Nginx (проксирование, gzip, security headers).
- **Настроить CI/CD (например, GitHub Actions): pipeline для запуска тестов, линтера и, возможно, сборки Docker-образа (бонус).**
- **Подготовить подробный отчёт: архитектурные решения, использованные метрики, результаты оценки (включая графики), анализ стоимости (время/токены), результаты статистических тестов. Экспортировать графики (PNG/SVG) для включения в отчёт.**
- Финальное тестирование всех контейнеров и сценариев.
- Обновить README (инструкция одной командой, архитектура, скриншоты, ссылка на отчёт).

**Результат:**  
Готовый проект, запускаемый одной командой через Docker, с UI для всех функций, автоматическими проверками и полной документацией.

**Критерии готовности:**
- [ ] Все страницы UI работают, включая просмотр логов.
- [ ] Docker Compose поднимает все сервисы.
- [ ] CI/CD pipeline настроен и проходит (если требуется бонус).
- [ ] Отчёт подготовлен и содержит все необходимые разделы.
- [ ] README содержит инструкцию.
- [ ] Демонстрация основных сценариев успешна.

---

## Общие замечания

- Каждый этап завершается коммитом и коротким отчётом.
- Рекомендуются ветки для каждого этапа (например, `week-1`).
- Тесты пишутся сразу (TDD приветствуется).
- В конце курса — презентация проекта.

**Удачи!** 🚀


## 💡 Примеры возможных проектных тем


### 1. ⚡ Прогноз потребления электроэнергии

**Задача:**  
Прогнозировать почасовое потребление электроэнергии на следующие 24–72 часа.

**Агенты:**
- **DataAnalyst** – ресемплинг до часовой частоты, выделение сезонностей (суточная, недельная), проверка выбросов.
- **ModelSelector** – выбор между SARIMA, Prophet, XGBoost (с экзогенными признаками: праздники, температура) через Tool Calling.
- **Validator** – проверка остатков на автокорреляцию (Ljung-Box), нормальность.
- **Critic** – оценивает, насколько модель адекватна текущему сезону и аномалиям.
- **Editor** – при необходимости корректирует гиперпараметры или добавляет признаки.

**База знаний:**  
Синтетические или реальные кейсы прогнозирования энергопотребления (например, из Monash Repository). Описания: тип ряда, лучшая модель, метрики.

**Данные:**  
`vitthalmadane/energy-consumption-time-series-dataset` (Kaggle) или Electricity Load Diagrams (UCI).

**Метрики:**  
MAE, RMSE, MASE, WAPE; Ljung-Box для остатков.

**HITL:**  
Если прогноз сильно расходится с историческими паттернами или Critic ставит низкий балл, оператор может вручную выбрать модель или горизонт.

---

### 2. 🛒 Прогноз спроса на товары (ритейл)

**Задача:**  
Прогнозировать дневные продажи для нескольких товаров/категорий на 7–30 дней вперёд.

**Агенты:**
- **DataAnalyst** – агрегация до дневной частоты, добавление календарных признаков, проверка иерархий.
- **ModelSelector** – выбор между ARIMA, Prophet, LightGBM (или комбинирование) через Tool Calling.
- **Validator** – проверка горизонта, отсутствие NaN, точность на последних точках.
- **Critic** – оценивает учёт промо-акций, праздников, трендов.
- **Editor** – предлагает добавить лаги, скользящие средние или экзогенные переменные.

**База знаний:**  
Кейсы прогнозирования продаж из M5 или синтетические. Справочник – термины (сезонность, тренд, промо).

**Данные:**  
M5 Forecasting (Kaggle), можно взять подмножество.

**Метрики:**  
WRMSSE (если используется M5), MAE, RMSE, MASE; статистические тесты для сравнения.

**HITL:**  
Если товар новый и данных мало, система может запросить у пользователя экспертные корректировки.

---

### 3. 🌤 Прогноз погоды (температура, осадки)

**Задача:**  
Прогнозировать среднесуточную температуру или количество осадков на 7–14 дней для конкретного города.

**Агенты:**
- **DataAnalyst** – работа с метеоданными, заполнение пропусков, выделение сезонности.
- **ModelSelector** – выбор между статистическими моделями и ML (XGBoost, LSTM – если добавить как бонус).
- **Validator** – проверка физической реалистичности прогноза (например, температура не выходит за климатические пределы).
- **Critic** – оценка точности на исторических аномалиях (волны жары, заморозки).
- **Editor** – добавление лагов давления, влажности и т.п.

**База знаний:**  
Кейсы прогнозирования погоды из Jena Climate, Monash. Справочник – метеорологические термины.

**Данные:**  
Jena Climate (Kaggle), `sumanthvrao/daily-climate-time-series-data`.

**Метрики:**  
MAE, RMSE, MAPE; сравнение с наивным прогнозом.

**HITL:**  
При экстремальных погодных условиях может потребоваться подтверждение эксперта.




### 4. 🏠 Прогноз стоимости недвижимости (на примере Республики Татарстан)

**Задача:**  
Прогнозировать рыночную стоимость квартиры или дома на основе набора признаков: площадь, этаж, район, год постройки, наличие парковки и т.д. Пользователь вводит параметры объекта — система выдаёт оценку и обоснование.

**Агенты:**
- **DataAnalyst** – проверяет типы признаков, обрабатывает пропуски, кодирует категориальные переменные (район, тип дома), строит базовую статистику. При `task_type == "tabular"` пропускает временную предобработку.
- **ModelSelector** – через Tool Calling выбирает между `fit_tabular_xgboost`, `fit_random_forest`, `fit_linear_regression`, учитывая результаты RAG-поиска похожих кейсов.
- **Validator** – проверяет, что предсказание не выходит за разумные пределы (например, цена не отрицательная), оценивает качество на отложенной выборке, отсутствие NaN.
- **Critic** – LLM-as-a-Judge оценивает, насколько выбранная модель адекватна для данного сегмента недвижимости, учтены ли важные факторы (локация, инфраструктура).
- **Editor** – при необходимости предлагает добавить полиномиальные признаки или взаимодействия (например, площадь × район).

**База знаний:**  
1000+ кейсов: описание объекта (признаки) → фактическая цена + лучшая модель и гиперпараметры. Справочник – термины рынка недвижимости, средние цены по районам, коэффициенты поправок. RAG находит похожие объекты для калибровки.

**Данные:**  
Открытые данные о ценах на недвижимость: Kaggle (например, датасеты по Казани), порталы объявлений (можно спарсить для учебных целей). Или синтетически сгенерировать с реалистичными распределениями.

**Метрики:**  
MAE, RMSE, MAPE (основные); R² (дополнительно). Для сравнения моделей – Diebold-Mariano (если применимо) или Wilcoxon signed-rank.

**HITL:**  
Если оценка сильно расходится с типичным диапазоном или Critic замечает, что не учтён важный фактор (например, близость к метро), запрос уходит на проверку риелтору-редактору, который может скорректировать цену или добавить комментарий.

---

### 5. 👥 Прогноз оттока клиентов (телеком / банк) — табличная классификация

**Задача:**  
Прогнозировать, уйдёт ли клиент в ближайшие 1–3 месяца (бинарная классификация) на основе его профиля: возраст, тариф, длительность обслуживания, сумма платежей, частота обращений в поддержку и т.д.

**Агенты:**
- **DataAnalyst** – анализирует пропуски, кодирует категории (тариф, регион), масштабирует числовые признаки, проверяет баланс классов.
- **ModelSelector** – через Tool Calling выбирает `fit_tabular_xgboost` или `fit_logistic_regression` (с учётом дисбаланса классов), используя RAG для подбора похожих кейсов оттока.
- **Validator** – проверяет, что предсказания – вероятности в диапазоне [0,1], отсутствуют NaN, оценивает ROC-AUC на валидации.
- **Critic** – оценивает, насколько модель интерпретируема и пригодна для бизнес-решения (важно объяснить, почему клиент уходит).
- **Editor** – может предложить добавить признаки (например, средний чек за последние 3 месяца) или изменить порог классификации.

**База знаний:**  
1000+ кейсов: набор признаков клиента → факт оттока + рекомендации по удержанию. Справочник – типовые профили уходящих клиентов, важные факторы. RAG помогает найти похожие сценарии.

**Данные:**  
Открытые датасеты оттока: Kaggle (Telecom Churn, Bank Customer Churn), можно синтетически расширить.

**Метрики:**  
Accuracy, F1, ROC-AUC (основные). Precision/Recall для оценки качества выявления уходящих.

**HITL:**  
Если модель с высокой вероятностью относит клиента к оттоку, но Critic сомневается (например, нестандартный профиль), маркетолог-редактор может подтвердить или уточнить решение. Результат сохраняется в базу.


### 6. 📈 Прогноз финансовых временных рядов (цены акций, криптовалют)

**Задача:**  
Прогнозировать цену закрытия финансового инструмента (акции, ETF, криптовалюта) на следующий день или несколько дней вперёд на основе исторических котировок и внешних факторов (новости, индексы, объёмы торгов).

**Агенты:**
- **DataAnalyst** – обработка пропусков, выбросов, расчёт технических индикаторов (скользящие средние, RSI, MACD), нормализация, проверка стационарности.
- **ModelSelector** – выбор между ARIMA/GARCH, Prophet, XGBoost, LSTM (бонус) через Tool Calling; при необходимости комбинирование моделей.
- **Validator** – проверка остатков на автокорреляцию и гетероскедастичность, адекватность волатильности, отсутствие look-ahead bias.
- **Critic** – оценивает, учтены ли рыночные режимы (тренд, флэт, высокая волатильность), нет ли переобучения.
- **Editor** – предлагает добавить лаги других инструментов, новостные сентименты, изменить гиперпараметры.

**База знаний:**  
Кейсы прогнозирования финансовых рядов (синтетические или реальные), описания: тип инструмента, частота, лучшая модель, метрики. Справочник – термины технического анализа, индикаторы, типовые паттерны. RAG помогает подбирать похожие исторические периоды.

**Данные:**  
Yahoo Finance (через `yfinance`), Kaggle (исторические данные криптовалют), MOEX. Можно синтетически генерировать ряды с заданными свойствами.

**Метрики:**  
MAE, RMSE, MAPE; directional accuracy (точность предсказания направления); Diebold-Mariano для сравнения моделей. Для оценки риска – Value at Risk (дополнительно).

**HITL:**  
Если прогноз противоречит фундаментальным событиям (например, отчёт о прибыли) или Critic считает модель неадекватной, трейдер-редактор может скорректировать или подтвердить решение.

---

### 7. 🚗 Прогноз интенсивности дорожного движения (транспорт)

**Задача:**  
Прогнозировать загруженность дорог или количество автомобилей на участке трассы на ближайшие часы/сутки на основе исторических данных о трафике, погоде, времени суток и праздниках.

**Агенты:**
- **DataAnalyst** – агрегация по времени, обработка пропусков, добавление календарных признаков (час, день недели, праздники), сезонная декомпозиция.
- **ModelSelector** – выбор между SARIMA, Prophet, LightGBM, возможно графовые модели (бонус) через Tool Calling.
- **Validator** – проверка физических ограничений (неотрицательность, максимум пропускной способности), остатков.
- **Critic** – оценивает, насколько модель учитывает пробки, аварии, погодные условия.
- **Editor** – добавляет лаги соседних участков, временные окна, взаимодействия.

**База знаний:**  
Кейсы прогнозирования трафика (Monash, UCI). Справочник – термины транспортного моделирования, типовые паттерны. RAG находит похожие дорожные сценарии.

**Данные:**  
UCI "Metro Interstate Traffic Volume", данные с датчиков (Caltrans PeMS), Kaggle. Можно взять открытые API городов.

**Метрики:**  
MAE, RMSE, MAPE, WAPE. Сравнение с наивным прогнозом (среднее по времени суток).

**HITL:**  
При аномальных событиях (ДТП, перекрытие) оператор может вручную скорректировать прогноз или добавить флаг события.

---

### 8. 🏥 Прогноз заболеваемости (эпидемиология)

**Задача:**  
Прогнозировать количество новых случаев заболевания (грипп, COVID-19) на неделю вперёд для региона на основе исторической динамики и внешних факторов (вакцинация, температура, плотность населения).

**Агенты:**
- **DataAnalyst** – работа с эпидемиологическими рядами, сглаживание, выделение сезонности, добавление экзогенных переменных.
- **ModelSelector** – выбор между SARIMA, Prophet, SEIR-подобными моделями (бонус), XGBoost через Tool Calling.
- **Validator** – проверка неотрицательности прогноза, сравнение с эпидемиологическими порогами, анализ остатков.
- **Critic** – оценивает, учтены ли меры карантина, вакцинация, сезонность.
- **Editor** – добавляет лаги заболеваемости в соседних регионах, полиномиальные тренды.

**База знаний:**  
Кейсы прогнозирования заболеваемости (синтетические, реальные данные). Справочник – эпидемиологические термины, инкубационные периоды, R0. RAG помогает находить похожие вспышки.

**Данные:**  
CDC FluView, данные по COVID-19 (Our World in Data, Kaggle), Monash. Можно синтетически моделировать.

**Метрики:**  
MAE, RMSE, MAPE; для оценки пиков – точность предсказания момента пика. Статистические тесты сравнения.

**HITL:**  
Если прогноз предсказывает резкий рост, эпидемиолог-редактор может подтвердить или скорректировать с учётом новых мер.

---

### 9. 🎓 Прогноз успеваемости студентов (траектории обучения) — табличная классификация/регрессия

**Задача:**  
Прогнозировать итоговую оценку студента по курсу (регрессия) или вероятность отчисления/неуспеваемости (бинарная классификация) на основе данных об активности: посещаемость, сдача домашних заданий, баллы за промежуточные тесты, время в LMS, социально-демографические признаки.

**Агенты:**
- **DataAnalyst** – обработка табличных данных: кодирование категорий (пол, специальность), импутация пропусков, масштабирование, анализ баланса классов.
- **ModelSelector** – через Tool Calling выбирает `fit_tabular_xgboost` или `fit_logistic_regression`, `fit_random_forest`, учитывая RAG-поиск похожих кейсов.
- **Validator** – проверка качества на кросс-валидации, отсутствие утечек, адекватность вероятностей.
- **Critic** – оценивает интерпретируемость модели (почему студент в группе риска), пригодность для принятия решений.
- **Editor** – предлагает добавить признаки (например, динамика оценок, активность в форуме), изменить порог.

**База знаний:**  
1000+ кейсов: профиль студента → итоговая оценка/статус + рекомендации. Справочник – факторы успеваемости, типовые паттерны. RAG находит похожих студентов для обоснования.

**Данные:**  
Открытые датасеты: Kaggle (Student Performance), Open University Learning Analytics Dataset, синтетические данные.

**Метрики:**  
Для регрессии: MAE, RMSE, R². Для классификации: Accuracy, F1, ROC-AUC. Анализ важности признаков.

**HITL:**  
Если модель предсказывает высокий риск отчисления, преподаватель-редактор может подтвердить или скорректировать (например, учесть личные обстоятельства). Результат сохраняется в базу.

---

### 10. 🏭 Прогноз отказов оборудования (предиктивное обслуживание) — временные ряды с сенсоров

**Задача:**  
Прогнозировать вероятность отказа промышленного оборудования (двигатель, турбина) в ближайшие N циклов на основе телеметрии с датчиков (вибрация, температура, давление).

**Агенты:**
- **DataAnalyst** – обработка многомерных временных рядов с датчиков, расчёт скользящих статистик, спектральных характеристик, сегментация.
- **ModelSelector** – выбор между классификаторами на основе извлечённых признаков (XGBoost, Random Forest) или рекуррентными сетями (LSTM, бонус) через Tool Calling.
- **Validator** – проверка вероятностей, отсутствие ложных срабатываний на валидации, анализ остатков.
- **Critic** – оценивает, насколько модель учитывает деградацию оборудования, не пропускает ли критические паттерны.
- **Editor** – предлагает добавить взаимодействия датчиков, временные лаги, изменить порог классификации.

**База знаний:**  
Кейсы отказов (NASA Turbofan, синтетические). Справочник – типовые неисправности, признаки износа. RAG находит похожие случаи.

**Данные:**  
NASA C-MAPSS (Kaggle), PHM Data Challenge, UCI. Можно синтетически генерировать.

**Метрики:**  
F1, Precision/Recall (важно минимизировать false negatives), ROC-AUC. Для регрессии оставшегося срока службы (RUL): MAE, RMSE.

**HITL:**  
Если модель предсказывает скорый отказ с высокой вероятностью, инженер-редактор может подтвердить (назначить внеплановое ТО) или отклонить, если это ложное срабатывание.
